In [16]:
import cv2
import numpy as np
import os

input_folder = "enhanced_frames"
output_folder = "output_crossing"

os.makedirs(output_folder, exist_ok=True)

def road_mask(gray):

    h, w = gray.shape

    mask = np.zeros((h, w), dtype=np.uint8)

    mask[int(h*0.45):h, int(w*0.05):int(w*0.95)] = 255

    return mask

def detect_faded_crossing(frame, output, mask):

    h, w = frame.shape[:2]

    roi = frame[int(h*0.45):h, :]

    roi_mask = mask[int(h*0.45):h, :]

    roi = cv2.bitwise_and(roi, roi, mask=roi_mask)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    blur = cv2.GaussianBlur(enhanced, (5,5), 0)

    _, binary = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)


    kernel = np.ones((5, 25), np.uint8)
    morph = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    stripe_count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 800:  
            continue

        x, y, wc, hc = cv2.boundingRect(c)

        aspect = wc / (hc + 1e-5)

        if aspect < 1.8:
            continue

        stripe_roi = gray[y:y+hc, x:x+wc]

        if stripe_roi.size == 0:
            continue

        mean_val = np.mean(stripe_roi)
        std_val = np.std(stripe_roi)

        if std_val < 10:
            continue

        if mean_val > 220:
            continue

        cv2.rectangle(
            output,
            (x, y + int(h*0.45)),
            (x + wc, y + hc + int(h*0.45)),
            (0, 165, 255),
            2
        )

        stripe_count += 1

    if stripe_count >= 2:
        cv2.putText(
            output,
            f"Faded Crossing ({stripe_count})",
            (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 165, 255),
            2
        )
        return stripe_count

    return 0

files = sorted(os.listdir(input_folder))

for file in files:

    if not file.lower().endswith(('.png','.jpg','.jpeg')):
        continue

    path = os.path.join(input_folder, file)

    frame = cv2.imread(path)

    if frame is None:
        continue

    output = frame.copy()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    mask = road_mask(gray)

    count = detect_faded_crossing(frame, output, mask)

    cv2.putText(
        output,
        f"Crossing: {count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imwrite(os.path.join(output_folder, file), output)

    print(f"{file} → detected stripes: {count}")

print("\n FADED CROSSING DETECTION COMPLETE")

AlligatorCracking_frame_0.jpg → detected stripes: 0
AlligatorCracking_frame_20.jpg → detected stripes: 0
AlligatorCracking_frame_35.jpg → detected stripes: 0
AlligatorCracking_frame_4.jpg → detected stripes: 0
AlligatorCracking_frame_51.jpg → detected stripes: 0
AlligatorCracking_frame_9.jpg → detected stripes: 0
EdgeCracking3_frame_72.jpg → detected stripes: 0
EdgeCracking4_frame_123.jpg → detected stripes: 0
EdgeCracking4_frame_129.jpg → detected stripes: 0
EdgeCracking4_frame_130.jpg → detected stripes: 0
EdgeCracking4_frame_133.jpg → detected stripes: 0
EdgeCracking4_frame_144.jpg → detected stripes: 0
EdgeCracking4_frame_154.jpg → detected stripes: 0
EdgeCracking9_frame_185.jpg → detected stripes: 0
EdgeCracking9_frame_202.jpg → detected stripes: 0
EdgeCracking9_frame_206.jpg → detected stripes: 0
EdgeCracking9_frame_217.jpg → detected stripes: 0
RoadPotholes2_frame_522.jpg → detected stripes: 0
RoadPotholes3_frame_575.jpg → detected stripes: 0
RoadPotholes3_frame_578.jpg → detect